# Task 4: Predictive Modeling

**BCG X Data Science Job Simulation on Forage - PowerCo churn case**

> **Simulation disclosure:** This notebook documents work completed as part of the **BCG X Data Science Job Simulation on Forage**. It is an educational job simulation and does not represent employment, an internship, or client work performed for BCG X or PowerCo.

## Objective
Train the Random Forest classifier specified in the simulation, evaluate its ability to identify churners, and interpret the strongest predictive features in the context of the original price-sensitivity hypothesis.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

# Shows plots in jupyter notebook
%matplotlib inline

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Load the model-ready dataset

In [3]:
df = pd.read_csv('../data/processed/data_for_predictions.csv')
df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)
df.head()

                                 id  cons_12m  cons_gas_12m  cons_last_month  \
0  24011ae4ebbe3035111d65fa7c15bc57  0.000000      4.739944         0.000000   
1  d29c2c54acc38ff3c0614d0a653813dd  3.668479      0.000000         0.000000   
2  764c75f661154dac3a6c254cd082ea7d  2.736397      0.000000         0.000000   
3  bba03439a292a1e166f80264c16191cb  3.200029      0.000000         0.000000   
4  149d57cf92fc41cf94415803a877cb4b  3.646011      0.000000         2.721811   

   forecast_cons_12m  forecast_discount_energy  forecast_meter_rent_12m  \
0           0.000000                       0.0                 0.444045   
1           2.280920                       0.0                 1.237292   
2           1.689841                       0.0                 1.599009   
3           2.382089                       0.0                 1.318689   
4           2.650065                       0.0                 2.122969   

   forecast_price_energy_off_peak  forecast_price_energy_peak  \
0  

## 2. Modeling setup

The simulation specifies a **Random Forest classifier**. This tree-based ensemble can capture nonlinear relationships and feature interactions without requiring feature standardization. Model performance is evaluated on a held-out test set because the primary business question is how well the model generalizes to unseen customers.


In [4]:
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

### Train/test split

Use a 75/25 split with `random_state=42` so the completed run is reproducible.

In [5]:
# Make a copy of our data
train_df = df.copy()

# Separate target variable from independent variables
y = df['churn']
X = df.drop(columns=['id', 'churn'])
print(X.shape)
print(y.shape)

(14606, 61)
(14606,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(10954, 61)
(10954,)
(3652, 61)
(3652,)


### Random Forest training

Train a baseline model with 100 estimators and a fixed random seed.

In [7]:
# Train the Random Forest classifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## 3. Model evaluation

Evaluate both class predictions and churn probabilities. Accuracy alone is insufficient because churners represent a small share of the customer base.

In [8]:
# Generate predictions on the test dataset
y_pred = model.predict(X_test)

# Generate probability of churn for each customer
y_pred_proba = model.predict_proba(X_test)[:, 1]

y_pred[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [9]:
# Calculate model performance metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9030668127053669
Precision: 0.7142857142857143
Recall: 0.0546448087431694
F1 Score: 0.10152284263959391
ROC-AUC: 0.6648810652245493

Confusion Matrix:
[[3278    8]
 [ 346   20]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      3286
           1       0.71      0.05      0.10       366

    accuracy                           0.90      3652
   macro avg       0.81      0.53      0.53      3652
weighted avg       0.89      0.90      0.86      3652



### Evaluation interpretation

The Random Forest achieved **90.3% accuracy**, but the class-level metrics show an important limitation:

- **Precision:** 71.4%
- **Recall:** 5.46%
- **F1 score:** 10.15%
- **ROC-AUC:** 66.49%

The confusion matrix shows that only **20 of 366 actual churners** in the test set were identified, while 346 churners were classified as retained customers.

For a retention use case, this baseline is therefore not operationally sufficient despite its high overall accuracy. The key improvement area is churn recall, ideally while preserving useful precision.


## 4. Feature importance

In [10]:
# Calculate feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Display the 10 most important features
feature_importance.head(10)

                    feature  importance
0                  cons_12m    0.051672
14               net_margin    0.049589
5   forecast_meter_rent_12m    0.049370
11     margin_gross_pow_ele    0.047671
12       margin_net_pow_ele    0.046966
3         forecast_cons_12m    0.046268
15                  pow_max    0.035858
2           cons_last_month    0.034973
49             months_activ    0.032355
10                 imp_cons    0.031554

### Feature-importance interpretation

The leading Random Forest features are `cons_12m`, `net_margin`, `forecast_meter_rent_12m`, gross and net power margins, forecast consumption, subscribed power, recent consumption, and customer activity duration.

Price-related variables do not appear among the ten most important features in this model. This does **not** establish that price has no effect on churn, and Random Forest feature importance should not be interpreted causally. It does, however, provide limited support for the hypothesis that price sensitivity is the primary churn driver in this dataset.


## 5. Business conclusion and next steps

- Do not use headline accuracy as the primary success measure for the churn model.
- Improve recall before using the model for operational retention targeting.
- Evaluate class-imbalance strategies and decision-threshold tuning in further modeling work.
- Combine predicted churn risk with customer commercial value when designing retention pilots.
- Test targeted interventions rather than assuming broad price discounts are the appropriate response for all customers.
